In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. MNIST WITHOUT PYTORCH NN MODULE

In [ ]:
'''
  Import the required dependencies.

'''

import torch
import numpy as np
import pandas as pd

In [ ]:
'''
  Check if the libraries are properly imported.

'''


print("PyTorch version: ", torch.__version__)
print("NumPy version: ", np.__version__)
print("Pandas version: ", pd.__version__)

In [ ]:
'''
  Check if GPU is available in the system. We will shift the model to GPU for faster training.

'''

gpu_aval = torch.cuda.is_available()
print(f"GPU available: {gpu_aval}")

if gpu_aval:
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

manual_seed = 42
torch.manual_seed(manual_seed)

print("Set random seed for torch as 42")

## 1. Building Dataloader Classes

In [ ]:
train = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

In [ ]:
from torch.utils.data import Dataset, DataLoader

'''
  PyTorch Dataset class to fetch items and other operation requirements.

'''

class MNISTDataset(Dataset):
    def __init__(self, dataset):
        self.data = torch.tensor(dataset.values, dtype=torch.float32)

    def __len__(self):
        return len(self.data) 

    def __getitem__(self, idx):
        item = self.data[idx]
        label = item[0]
        pixels = item[1:]
        pixels = pixels / 255
        return pixels, label

In [ ]:
from torch.utils.data import random_split

'''
  Creating train and test dataloader to facililate the training and testing process

'''

full_dataset = MNISTDataset(dataset=train)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=8
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8
)

print(f"Total samples: {len(full_dataset)}")
print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")

In [ ]:
'''
  Creating an iterator from train_loader dataloader

'''

data_iterator = iter(train_loader)

pixels, label = next(data_iterator)

print(f"Pixels tensor shape: {pixels.shape}")
print(f"Pixels tensor dtype: {pixels.dtype}")
print(f"Label tensor shape: {label.shape}")
print(f"Label tensor dtype: {label.dtype}")

In [ ]:
sample_iter = iter(train_loader)

In [ ]:
len(pixels)

In [ ]:
pixels, label = next(sample_iter)

In [ ]:
pixels[0]

In [ ]:
label[0]

## 2. Training Functions Implementation

In [ ]:
epochs = 104
EPOCHS_PER_EVAL = 10
lr = 1e-2
hidden_features = 512

w1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, 784)).to(device).requires_grad_()
b1 = torch.distributions.Uniform(-np.sqrt(1/784), np.sqrt(1/784)).sample((hidden_features, )).to(device).requires_grad_()
w2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features, hidden_features)).to(device).requires_grad_()
b2 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((hidden_features, )).to(device).requires_grad_()
w3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, hidden_features)).to(device).requires_grad_()
b3 = torch.distributions.Uniform(-np.sqrt(1/hidden_features), np.sqrt(1/hidden_features)).sample((10, )).to(device).requires_grad_()

In [ ]:
ep = 1e-6

def relu(x):
    return x.clamp(min=0)

def batch_norm(x, gamma):
    batch_mean = x.mean(dim=0)
    batch_var = (x-batch_mean).var(dim=0)
    x_norm = (x-batch_mean) / (torch.sqrt(batch_var) + ep)

def linear(x, w, b):
    return x.matmul(w.t()) + b

def sgd_step(params, lr):
    with torch.no_grad():
        for p in params:
            if p.grad is not None:
                p -= lr * p.grad
                p.grad.zero_()

def softmax(x):
    return x - torch.logsumexp(x, dim=1, keepdim=True)

def forward(x):
    z1 = linear(x, w1, b1)
    a1 = relu(z1)
    z2 = linear(a1, w2, b2)
    a2 = relu(z2)
    z3 = linear(a2, w3, b3)

    return z3



In [ ]:
def loss_function(log_probs, targets):
    return -log_probs[torch.arange(targets.size(0), device=targets.device), targets].mean()

In [ ]:
params = [w1, b1, w2, b2, w3, w3]

In [ ]:
def compute_loss_and_acc(x, y):
    logits = forward(x)
    logp = softmax(logits)
    loss = loss_function(logp, y)
    preds = logp.argmax(dim=1)
    acc = (preds == y).float().mean()
    return loss, acc

## 3. Training Pipeline

In [ ]:
print(f"lr={lr}")
seen = 0

for epoch in range(1, epochs + 1):
    total_loss = 0.0
    total_acc = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device).long()
    
        logits = forward(xb)
        logp = softmax(logits)
        loss = loss_function(logp, yb)

        loss.backward()

        sgd_step(params, lr)

        bs = xb.shape[0]
        total_loss += loss.item() * bs
        total_acc += (logp.argmax(dim=1) == yb).float().sum().item()
        seen += bs

    train_loss = total_loss / seen
    train_acc  = total_acc / seen

    total_loss = 0.0
    total_acc = 0.0
    seen = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device).long()
            loss, acc = compute_loss_and_acc(xb, yb)
            bs = xb.shape[0]
            total_loss += loss.item() * bs
            total_acc += acc.item() * bs
            seen += bs
    test_loss = total_loss / seen
    test_acc  = total_acc / seen

    print(f"Epoch {epoch:2d} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"test loss {test_loss:.4f} acc {test_acc:.4f}")
